In [19]:
import pandas as pd
import os
from sqlalchemy import create_engine

# SQL Server connection
server = r".\SQLEXPRESS01"
database = "NYC_Taxi_Analytics"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
)

engine = create_engine(
    connection_string,
    fast_executemany=True
)

# Cleaned data folder
cleaned_folder = "cleaned_data"

cleaned_files = sorted(
    file for file in os.listdir(cleaned_folder)
    if file.endswith(".parquet")
)

total_loaded = 0

for file in cleaned_files:

    print(f"Loading {file}...")

    file_path = os.path.join(cleaned_folder, file)

    df = pd.read_parquet(file_path)

    df = df.rename(columns={
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "PULocationID": "pickup_location_id",
        "DOLocationID": "dropoff_location_id"
    })

    df = df[
        [
            "pickup_datetime",
            "dropoff_datetime",
            "trip_distance",
            "pickup_location_id",
            "dropoff_location_id",
            "fare_amount",
            "total_amount",
            "trip_duration_minutes",
            "pickup_date",
            "pickup_hour",
            "day_name",
            "day_type",
            "avg_speed_mph"
        ]
    ]

    df.to_sql(
        "taxi_trips",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=25000,
        method=None
    )

    total_loaded += len(df)

    print(f"  Rows loaded: {len(df):,}")

print(f"\nTotal rows loaded: {total_loaded:,}")

Loading yellow_tripdata_2025-01_clean.parquet...
  Rows loaded: 3,327,575
Loading yellow_tripdata_2025-02_clean.parquet...
  Rows loaded: 3,388,311
Loading yellow_tripdata_2025-03_clean.parquet...
  Rows loaded: 3,912,535
Loading yellow_tripdata_2025-04_clean.parquet...
  Rows loaded: 3,747,243
Loading yellow_tripdata_2025-05_clean.parquet...
  Rows loaded: 4,200,532
Loading yellow_tripdata_2025-06_clean.parquet...
  Rows loaded: 3,977,288
Loading yellow_tripdata_2025-07_clean.parquet...
  Rows loaded: 3,594,433
Loading yellow_tripdata_2025-08_clean.parquet...
  Rows loaded: 3,263,664
Loading yellow_tripdata_2025-09_clean.parquet...
  Rows loaded: 3,939,423
Loading yellow_tripdata_2025-10_clean.parquet...
  Rows loaded: 4,037,616
Loading yellow_tripdata_2025-11_clean.parquet...
  Rows loaded: 3,722,494
Loading yellow_tripdata_2025-12_clean.parquet...
  Rows loaded: 4,196,482

Total rows loaded: 45,307,596
